# Sports Betting Arbitrage Scraper (Novibet, Stoiximan, Efbet)
This notebook scrapes odds from three major providers using direct deep links, cleans the data, and identifies arbitrage opportunities.

In [ ]:
import sys
sys.path.append('..')

# Import existing modules
try:
    from web_scrape_functions import novibet_functions as nv
    from web_scrape_functions import stoiximan_function as stm
except ImportError:
    print("Warning: web_scrape_functions folder or modules not found.")

import pandas as pd
import duckdb
import time
import re
from unidecode import unidecode
from fuzzywuzzy import fuzz
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [ ]:
# --- SETUP DRIVER ---
options = webdriver.ChromeOptions()
# options.add_argument("--headless") # Uncomment for background run
options.add_argument("--window-size=1920,1200")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 15)

## 1. Scraping Data

In [ ]:
# --- 1.1 NOVIBET ---
print("--- Starting Novibet ---")
try:
    page_url = 'https://www.novibet.gr/en/sports'
    
    # Football
    football_string = nv.novibet_football_text(page_url, driver)
    nv.novibet_football_export(football_string)
    print("Novibet Football saved.")

    # Basketball
    basketball_string = nv.novibet_basketball_text(driver)
    nv.novibet_basketball_export(basketball_string)
    print("Novibet Basketball saved.")
    
except Exception as e:
    print(f"Novibet Error: {e}")

In [ ]:
# --- 1.2 STOIXIMAN ---
print("\n--- Starting Stoiximan ---")
try:
    # Football
    football_url = 'https://en.stoiximan.gr/sport/soccer/'
    football_string_stm = stm.stoiximan_football_text(football_url, driver)
    stm.stoiximan_football_export(football_string_stm)
    print("Stoiximan Football saved.")

    # Basketball
    basketball_url = 'https://en.stoiximan.gr/sport/basketball/'
    basketball_string_stm = stm.stoiximan_basketball_text(basketball_url, driver)
    stm.stoiximan_basketball_export(basketball_string_stm)
    print("Stoiximan Basketball saved.")
    
except Exception as e:
    print(f"Stoiximan Error: {e}")

In [ ]:
# --- 1.3 EFBET (Updated with Correct Links) ---
print("\n--- Starting Efbet ---")

def scrape_efbet_sport(driver, sport_url, sport_name):
    print(f"Navigating to {sport_name}...")
    driver.get(sport_url)
    time.sleep(5) # Allow SPA routing to load
    
    # Accept Cookies if present
    try:
        wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Accept')]"))).click()
    except:
        pass
    
    # Apply 24h Filter (Essential for arbitrage)
    try:
        wait.until(EC.element_to_be_clickable((By.XPATH, "//span[contains(text(), '24h') or contains(text(), 'Today')]"))).click()
        time.sleep(2)
    except:
        print("24h filter not found (or already active).")

    # Infinite Scroll
    last_height = driver.execute_script("return document.body.scrollHeight")
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height
        
    # Extract Text
    try:
        container = driver.find_element(By.CSS_SELECTOR, "div.center-view-content")
        return container.text
    except Exception as e:
        print(f"Error getting text for {sport_name}: {e}")
        return ""

def export_efbet_football(text_data):
    lines = text_data.split('\n')
    # Remove Headers/Garbage
    garbage = ['Soccer', 'All', '1', 'X', '2', 'Over', 'Under', 'BTS', 'Double Chance', 'Draw No Bet', 'Winner', 'Handicap']
    clean = [x for x in lines if x not in garbage and len(x) > 1]
    
    # Find Matches (Time Pattern: 14:00 or Date 24/03)
    indices = [i for i, x in enumerate(clean) if (':' in x and len(x) < 6 and x[0].isdigit()) or '/' in x]
    
    data = []
    if indices:
        # Chunk list into match rows
        chunks = [clean[i:j] for i, j in zip(indices, indices[1:] + [len(clean)])]
        
        for chunk in chunks:
            # Pad chunk to ensure 15 columns (standard structure)
            if len(chunk) < 15: chunk.extend(['No_bet'] * (15 - len(chunk)))
            data.append(chunk[:15])
            
    df = pd.DataFrame(data, columns=['Time', 'Team1', 'Team2', '1', 'X', '2', 'O_odds', 'U_odds', 
                                     'm1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7'])
    # Convert to numeric
    for c in ['1', 'X', '2', 'O_odds', 'U_odds']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
        
    df.to_csv('data/efbet_football.csv', index=False)
    print("Efbet Football saved to CSV.")

def export_efbet_basketball(text_data):
    lines = text_data.split('\n')
    garbage = ['Basketball', 'Winner', 'Handicap', 'Over/Under', 'Total', '1', '2']
    clean = [x for x in lines if x not in garbage and len(x) > 1]
    
    indices = [i for i, x in enumerate(clean) if ':' in x and len(x) < 6]
    
    data = []
    if indices:
        chunks = [clean[i:j] for i, j in zip(indices, indices[1:] + [len(clean)])]
        for chunk in chunks:
            if len(chunk) < 12: chunk.extend(['No_bet'] * (12 - len(chunk)))
            data.append(chunk[:12])
            
    df = pd.DataFrame(data, columns=['Time', 'Team1', 'Team2', '1', '2', 'Hand_Val', 'H1', 'H2', 'Tot_Val', 'O', 'U', 'm1'])
    df.to_csv('data/efbet_basketball.csv', index=False)
    print("Efbet Basketball saved to CSV.")

# EXECUTE EFBET SCRAPE
try:
    # Correct Links
    efbet_foot_url = 'https://www.efbet.gr/en/sports/pre-match/event-view/Soccer'
    efbet_bsk_url = 'https://www.efbet.gr/en/sports/pre-match/event-view/Basketball'
    
    # Scrape & Export
    foot_text = scrape_efbet_sport(driver, efbet_foot_url, "Soccer")
    if foot_text: export_efbet_football(foot_text)
    
    bsk_text = scrape_efbet_sport(driver, efbet_bsk_url, "Basketball")
    if bsk_text: export_efbet_basketball(bsk_text)
    
except Exception as e:
    print(f"Efbet Main Execution Error: {e}")

## 2. Data Cleaning & Queries (3-Way)

In [ ]:
def clean_df(df):
    if df is None or df.empty: return pd.DataFrame()
    # Standardize columns
    if 'One_odd' in df.columns: df.rename(columns={'One_odd': '1', 'X_odd': 'X', 'Two_odd': '2', 'O_odd': 'O_odds', 'U_odd': 'U_odds'}, inplace=True)
    
    # Clean Team Names
    cols = ['Team1', 'Team2']
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype(str).apply(lambda x: unidecode(x).lower())
            # Remove extra words like 'fc', 'single words'
            df[c] = df[c].apply(lambda x: ' '.join([w for w in x.split() if len(w) > 2]))
    return df

# Load & Clean
try:
    df_novi = clean_df(pd.read_csv('data/novibet_football.csv'))
    df_stoi = clean_df(pd.read_csv('data/stoiximan_football.csv'))
    df_ef = clean_df(pd.read_csv('data/efbet_football.csv'))
except FileNotFoundError:
    print("One or more CSVs missing. Check scraping steps.")
    df_novi, df_stoi, df_ef = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

## 3. SQL Queries (DuckDB)
We use SQL to join the tables based on fuzzy matching logic or standard names.

In [ ]:
conn = duckdb.connect()

# Register Tables
if not df_novi.empty: conn.register('t1', df_novi)
if not df_stoi.empty: conn.register('t2', df_stoi)
if not df_ef.empty: conn.register('t3', df_ef)

# --- 3-Way Over/Under Query ---
query_ou = """
SELECT 
    t1.Team1, t1.Team2,
    t1.O_odds as O_novi, t2.O_odds as O_stoi, t3.O_odds as O_ef,
    t1.U_odds as U_novi, t2.U_odds as U_stoi, t3.U_odds as U_ef,
    GREATEST(t1.O_odds, t2.O_odds, t3.O_odds) as O_Max,
    GREATEST(t1.U_odds, t2.U_odds, t3.U_odds) as U_Max,
    (1/GREATEST(t1.O_odds, t2.O_odds, t3.O_odds) + 1/GREATEST(t1.U_odds, t2.U_odds, t3.U_odds)) as Arb
FROM t1
JOIN t2 ON t1.Team1 = t2.Team1 -- Simple join (requires clean names)
JOIN t3 ON t1.Team1 = t3.Team1
WHERE Arb < 1.00
ORDER BY Arb ASC
"""

try:
    print("--- Arbitrage Results (SQL Exact Match) ---")
    result = conn.execute(query_ou).df()
    display(result)
except Exception as e:
    print(f"SQL Error (tables might be empty): {e}")

## 4. Fuzzy Matching (Advanced)
If SQL returns few results due to name mismatch (e.g. 'Olympiakos' vs 'Olympiacos'), use this fuzzy loop.

In [ ]:
matches = []
print("Running Fuzzy Match...")

# Iterate Novibet
for i, row in df_novi.iterrows():
    t1 = row['Team1']
    t2 = row['Team2']
    
    # Match Stoiximan
    m_stoi = df_stoi[
        (df_stoi['Team1'].apply(lambda x: fuzz.token_sort_ratio(x, t1)) > 85) &
        (df_stoi['Team2'].apply(lambda x: fuzz.token_sort_ratio(x, t2)) > 85)
    ]
    
    # Match Efbet
    m_ef = df_ef[
        (df_ef['Team1'].apply(lambda x: fuzz.token_sort_ratio(x, t1)) > 85) &
        (df_ef['Team2'].apply(lambda x: fuzz.token_sort_ratio(x, t2)) > 85)
    ]
    
    if not m_stoi.empty or not m_ef.empty:
        # Get Odds (Handle missing providers with 0)
        o_novi = row.get('O_odds', 0)
        u_novi = row.get('U_odds', 0)
        
        o_stoi = m_stoi.iloc[0]['O_odds'] if not m_stoi.empty else 0
        u_stoi = m_stoi.iloc[0]['U_odds'] if not m_stoi.empty else 0
        
        o_ef = m_ef.iloc[0]['O_odds'] if not m_ef.empty else 0
        u_ef = m_ef.iloc[0]['U_odds'] if not m_ef.empty else 0
        
        # Calc Max
        max_o = max(o_novi, o_stoi, o_ef)
        max_u = max(u_novi, u_stoi, u_ef)
        
        if max_o > 1 and max_u > 1:
            arb = (1/max_o) + (1/max_u)
            if arb < 1.0:
                matches.append({
                    'Match': f"{t1} vs {t2}",
                    'Arb': arb,
                    'ROI': (1-arb)*100,
                    'Max_O': max_o, 'Max_U': max_u,
                    'Bookie_O': 'Novibet' if max_o==o_novi else ('Stoiximan' if max_o==o_stoi else 'Efbet'),
                    'Bookie_U': 'Novibet' if max_u==u_novi else ('Stoiximan' if max_u==u_stoi else 'Efbet')
                })

df_res = pd.DataFrame(matches).sort_values('Arb')
print("Fuzzy Arbitrage Opportunities:")
display(df_res)

driver.quit()